In [1]:
import sys
import pathlib

# Locate the repo root
_search = pathlib.Path.cwd()
for _ in range(8):
    if (_search / "src" / "qdk_qualtran_comparison").is_dir():
        REPO_ROOT = str(_search / "src")
        break
    _search = _search.parent
else:
    REPO_ROOT = str(pathlib.Path.cwd())

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.4g}".format)

import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# ── Pipeline core modules ─────────────────────────────────────────────────────
from qdk_qualtran_comparison.config import (
    PipelineConfig,
    TranspileConfig,
    AzureConfig,
    QualtranConfig,
)
from qdk_qualtran_comparison.circuit.transpile import (
    transpile_to_clifford_t,
    circuit_stats,
    circuit_to_qasm,
)
from qdk_qualtran_comparison.compare.metrics import compare, enrich_from_circuit

# ── Bridge helper (inject Azure params -> Qualtran config) ─────────────────────
from qdk_qualtran_comparison.estimators.azure import apply_azure_to_qualtran

# ── Estimators (imported lazily inside run_estimation per circuit) ─────────────

print(f"REPO_ROOT: {REPO_ROOT}")
print("Imports OK")

REPO_ROOT: /Users/kevinli/Documents/resourceEstimation/qdk-qualtran-comparison/src
Imports OK


In [2]:
cfg = PipelineConfig(
    transpile=TranspileConfig(
        optimization_level=1,
        seed_transpiler=42,
        rotation_synthesis_enabled=False,
        rotation_synthesis_epsilon=1e-4,
        synthesis_strategy="synth",
        synthesis_method='bqskit',
    ),
    azure=AzureConfig(
        error_budget=0.01,
        error_rate=1e-3,
        gate_time_ns=50.0,
        measurement_time_ns=100.0,
        factory_type="RoundBased",
        slow_down_factors=[1.0],
        optimization_level=1,
        use_graph=True,
        use_qualtran_parameters=True,
        minimize="qubit_hours",
        pareto_index=0,
    ),
    qualtran=QualtranConfig(
        data_d=23,
        phys_err=1e-3,
        error_budget=0.01,
        data_block="fast",
        factory_type="15to1",
        n_factories=6,
        use_gidney_fowler=False,
        use_beverland=True,
        use_azure_parameters=True,
        pareto_index=0,
    ),
)

print(cfg)

PipelineConfig(hamlib=HamlibConfig(hdf5_path='./../hamlib/condensedmatter/heisenberg/heis.hdf5', key=None, key_index=313), evolution=EvolutionConfig(evolution_time=1.0, synthesis_order=2, synthesis_reps=10), transpile=TranspileConfig(basis_gates=['cx', 'cz', 'h', 's', 'sdg', 'swap', 'x', 'y', 'z', 't', 'tdg', 'rz'], optimization_level=1, seed_transpiler=42, rotation_synthesis_enabled=False, rotation_synthesis_epsilon=0.0001, synthesis_strategy='synth', synthesis_method='bqskit', pygridsynth_precision=None), azure=AzureConfig(error_budget=0.01, error_rate=0.001, gate_time_ns=50.0, measurement_time_ns=100.0, two_qubit_gate_time_ns=None, code_distance=None, factory_type='RoundBased', slow_down_factors=[1.0], optimization_level=1, use_graph=True, use_qualtran_parameters=True, minimize='qubit_hours', pareto_index=0), qualtran=QualtranConfig(data_d=23, data_d_sweep=None, phys_err=0.001, t_gate_ns=50.0, t_meas_ns=100.0, cycle_time_us=1.0, error_budget=0.01, data_block='fast', factory_type='15

In [3]:
# ---------------------------------------------------------------------------
# Helper: load a circuit from a .qasm file
# ---------------------------------------------------------------------------

def load_circuit_from_file(path: str):
    """Load a quantum circuit from an OpenQASM file.

    Supports OpenQASM 2.0 and 3.0 files.

    Parameters
    ----------
    path : str
        Path to a .qasm or .qasm3 file

    Returns
    -------
    QuantumCircuit
    """
    from qiskit import QuantumCircuit, qasm3

    if path.endswith(".qasm3"):
        return qasm3.load(path)
    else:
        return QuantumCircuit.from_qasm_file(path)

from pathlib import Path

# Define which circuit group to run
qasm_dir = Path("../01_circuit_generation/qasm3_circuits/cube/625_1000_173")

c_count, q_count, t_count = map(
    int,
    qasm_dir.name.split("_")
)

print(c_count)
print(q_count)
print(t_count)

circuit_list = sorted(
    str(path)
    for path in qasm_dir.glob("*.qasm3")
)

print(f"Found {len(circuit_list)} QASM circuits")
print(circuit_list[:5])

625
1000
173
Found 625 QASM circuits
['../01_circuit_generation/qasm3_circuits/cube/625_1000_173/1000_1.qasm3', '../01_circuit_generation/qasm3_circuits/cube/625_1000_173/1000_101.qasm3', '../01_circuit_generation/qasm3_circuits/cube/625_1000_173/1000_108.qasm3', '../01_circuit_generation/qasm3_circuits/cube/625_1000_173/1000_116.qasm3', '../01_circuit_generation/qasm3_circuits/cube/625_1000_173/1000_123.qasm3']


In [4]:
import re

def _safe_get(result, attr, default=None):
    """Safely retrieve an attribute from a possibly-None result."""
    if result is None:
        return default
    return getattr(result, attr, default)


def transpile_circuit(qc, cfg):
    """Transpile a single circuit to Clifford+T.

    Returns (clifford_t_circuit, stats_dict) or (None, {}) on failure.
    """
    try:
        ct = transpile_to_clifford_t(qc, cfg.transpile)
        stats = circuit_stats(ct)
        return ct, stats
    except Exception as exc:
        print(f"  Warning: Transpilation failed: {exc}")
        return None, {}


def run_estimation(ct_circuit, config):
    """Run one Azure and one Qualtran estimate on a single transpiled circuit.

    Returns (azure_result, qualtran_result) - each may be None on failure.
    Both results are enriched with circuit-derived metrics via enrich_from_circuit().
    """
    azure_result = None
    qualtran_result = None

    # -- Azure QDK ---------------------------------------------------------------
    try:
        from qdk_qualtran_comparison.estimators.azure import estimate as azure_estimate
        azure_result = azure_estimate(ct_circuit, config)
        azure_result = enrich_from_circuit(azure_result, ct_circuit)
    except ImportError as e:
        print(f"  Warning: Azure unavailable: {e}")
    except Exception as e:
        print(f"  Warning: Azure estimation failed: {e}")

    # -- Bridge: inject Azure params -> Qualtran config --------------------------
    if azure_result is not None and config.qualtran.use_azure_parameters:
        az_d = _safe_get(azure_result, 'code_distance')
        az_n = _safe_get(azure_result, 'num_factories')
        if (az_d is not None or az_n is not None):
            try:
                config = apply_azure_to_qualtran(azure_result, config)
            except Exception as e:
                print(f"  Warning: Azure->Qualtran bridge failed: {e}")

    # -- Qualtran ----------------------------------------------------------------
    try:
        from qdk_qualtran_comparison.estimators.qualtran import estimate as qt_estimate
        qualtran_result = qt_estimate(ct_circuit, config)
        qualtran_result = enrich_from_circuit(qualtran_result, ct_circuit)
    except ImportError as e:
        print(f"  Warning: Qualtran unavailable: {e}")
    except Exception as e:
        print(f"  Warning: Qualtran estimation failed: {e}")

    return azure_result, qualtran_result


# ---------------------------------------------------------------------------
# Simplify estimator names for plotting.
# The full name (with config params) is preserved in the DataFrame; this helper
# only extracts the family so that all points from the same estimator produce
# a single continuous line on each plot.
# ---------------------------------------------------------------------------

_FAMILY_RE = re.compile(r"^(Azure|Qualtran)")


def _family(name: str) -> str:
    """Return 'Azure' or 'Qualtran', falling back to the full name."""
    m = _FAMILY_RE.search(name)
    return m.group(1) if m else name


# ---------------------------------------------------------------------------
# collect_metrics: one row per (circuit, estimator) — same format as
# comparison_simple.ipynb but extended with qubit breakdown columns.
# ---------------------------------------------------------------------------

def collect_metrics(circuit_name, azure_result, qualtran_result, ct_stats):
    """Collect metrics into DataFrame rows.

    Keeps the side-by-side format from comparison_simple.ipynb:
      Metric | Azure (d=...,...) | Qualtran (d=...,...) | Ratio
    for every circuit, plus qubit breakdown columns for plotting.

    Missing fields become None -> pandas renders them blank.
    """
    for result in [azure_result, qualtran_result]:
        if result is None:
            continue
        t_count = _safe_get(result, 't_count')
        runtime = _safe_get(result, 'runtime_seconds')
        total_q = _safe_get(result, 'physical_qubits')
        compute_q = _safe_get(result, 'physical_compute_qubits')
        factory_q = _safe_get(result, 'physical_factory_qubits')

        # Space-time volume (qubit-seconds) using the same units each estimator reports.
        if total_q is not None and runtime is not None:
            space_time = float(total_q) * runtime
        else:
            space_time = None

        yield {
            "circuit_name":       circuit_name,
            "estimator_family":   _family(result.estimator_name),
            "t_count":            t_count,
            "clifford_count":     _safe_get(result, 'clifford_count'),
            "rotation_count":     _safe_get(result, 'rotation_count'),
            "toffoli_count":      _safe_get(result, 'toffoli_count'),
            "measurement_count":  _safe_get(result, 'measurement_count'),
            "runtime_seconds":    runtime,
            "total_qubits":       total_q,
            "compute_qubits":     compute_q,
            "factory_qubits":     factory_q,
            "space_time_volume":  space_time,
            "code_distance":      _safe_get(result, 'code_distance'),
            "logical_error_rate": _safe_get(result, 'logical_error_rate'),
            "error_budget":       _safe_get(result, 'error_budget'),
            "physical_error_rate":_safe_get(result, 'physical_error_rate'),
            "logical_qubits":     _safe_get(result, 'logical_qubits'),
            "logical_cycles":     _safe_get(result, 'logical_cycles'),
            "factory_count":      _safe_get(result, 'factory_count'),
            "num_factories":      _safe_get(result, 'num_factories'),
        }


def run_benchmark(circuit_list, config):
    """Run the full multi-circuit benchmarking pipeline.

    Parameters
    ----------
    circuit_list : list[QuantumCircuit] or list[str]  circuits or file paths
    config       : PipelineConfig

    Returns
    -------
    (pandas.DataFrame, dict[str, dict[str, EstimationResult]])
        DataFrame: one row per (circuit, estimator) — same format as
        comparison_simple.ipynb extended with qubit breakdown columns.
        circuit_results: nested dict {circuit_name: {'Azure': result, 'Qualtran': result}}
            so the side-by-side section can use the real EstimationResult objects.

        Columns for plotting:
            circuit_name, estimator_family, t_count, total_qubits,
            compute_qubits, factory_qubits, space_time_volume, runtime_seconds
        Columns from the comparison layer:
            clifford_count, rotation_count, logical_error_rate, code_distance, etc.
    """
    all_rows = []
    circuit_results = {}  # {circuit_name: {'Azure': result, 'Qualtran': result}}

    for idx, item in enumerate(circuit_list):
        # Derive circuit name
        if isinstance(item, str):
            circuit_name = pathlib.Path(item).stem
        elif hasattr(item, 'name') and getattr(item, 'name', None):
            circuit_name = item.name
        else:
            circuit_name = f"circuit_{idx}"

        # Step A: load (if path string) -> transpile
        qc = load_circuit_from_file(item) if isinstance(item, str) else item
        ct_circuit, ct_stats = transpile_circuit(qc, config)
        if ct_circuit is None:
            print(f"  Skipping circuit '{circuit_name}' (transpilation failed).")
            continue

        print(f"[{idx+1}/{len(circuit_list)}] Circuit '{circuit_name}': "
              f"T={ct_stats.get('t_count', '?')}, qubits={ct_stats.get('num_qubits', '?')}")

        # Step B: estimate (one Azure + one Qualtran)
        azure_res, qualtran_res = run_estimation(ct_circuit, config)

        # Save real EstimationResult objects for side-by-side comparison.
        circuit_results[circuit_name] = {}
        if azure_res is not None:
            circuit_results[circuit_name]['Azure'] = azure_res
        if qualtran_res is not None:
            circuit_results[circuit_name]['Qualtran'] = qualtran_res

        # Step C: collect metrics into DataFrame rows
        rows = list(collect_metrics(circuit_name, azure_res, qualtran_res, ct_stats))
        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)
    return df, circuit_results

In [ ]:
import time

start_time = time.time()
benchmark_df, circuit_results = run_benchmark(circuit_list, cfg)
end_time = time.time()
exec_time = end_time - start_time
print(f"\nBenchmark complete ({exec_time:.4f} seconds). {len(benchmark_df)} rows collected.")

# -- Display side-by-side comparison for the first circuit -------------------
# Uses the real EstimationResult objects saved during benchmarking — the same
# approach as comparison_simple.ipynb (cells with report, comparison_dataframe, etc.).
from qdk_qualtran_comparison.compare.metrics import compare
from qdk_qualtran_comparison.compare.tables import (
    comparison_dataframe, differences_dataframe, missing_dataframe,
)

estimator_names_in_df = benchmark_df['estimator_family'].unique()
circuits_list = benchmark_df['circuit_name'].unique()

if len(circuits_list) > 0 and len(estimator_names_in_df) >= 2:
    first_circuit = circuits_list[0]

    # Pull the real EstimationResult objects (not fake wrappers).
    results_map = circuit_results.get(first_circuit, {})
    azure_r = results_map.get('Azure')
    qt_r = results_map.get('Qualtran')

    available = [r for r in [azure_r, qt_r] if r is not None]

    if len(available) >= 2:
        # Build comparison report using the same helper as comparison_simple.ipynb.
        report = compare(available)
        print(f"\nCircuit : {first_circuit}")
        print(f"Estimators compared   : {report.estimator_names}")
        print(f"Shared metrics        : {len(report.shared_metrics)}")
        print(f"N/A in ≥1 estimator   : {len(report.missing_metrics)}")
        print(f"Numeric differences   : {len(report.differences)}")

        # Full comparison table — identical format to comparison_simple.ipynb.
        n_total = len(report.metric_rows)
        display(Markdown(
            f"*{n_total} metrics total — "
            f"**{len(report.shared_metrics)} shared** | "
            f"**{len(report.missing_metrics)} framework-specific*.*"
        ))
        df_comp = comparison_dataframe(report)
        display(df_comp)

        # Metrics that differ between estimators.
        diff_df = differences_dataframe(report)
        if not diff_df.empty:
            print("\nMetrics that differ:")
            display(diff_df)

    else:
        print("Need both Azure and Qualtran results for comparison.")
else:
    print("Not enough data for a side-by-side table.")

[transpile] Passthrough mode produced 0 rotation gates. The input circuit may already be in a Clifford+T basis, or all rotations were cancelled during optimization. This is expected only if the Hamiltonian has no non-Clifford Trotter terms.


[transpile] Passthrough mode: preserving rotation gates (synthesis disabled).
[transpile] Passthrough result (rotations preserved): 0 rotation gate(s) (none)
[transpile] Passthrough done: rotation_gates=0  T_gates=1  depth=3
[transpile] WARNING: 0 rotation gates in passthrough output. If you expected Rz gates, check basis_gates and optimization_level.
[1/625] Circuit '1000_1': T=1, qubits=1000


[transpile] Passthrough mode produced 0 rotation gates. The input circuit may already be in a Clifford+T basis, or all rotations were cancelled during optimization. This is expected only if the Hamiltonian has no non-Clifford Trotter terms.


[transpile] Passthrough mode: preserving rotation gates (synthesis disabled).
[transpile] Passthrough result (rotations preserved): 0 rotation gate(s) (none)
[transpile] Passthrough done: rotation_gates=0  T_gates=101  depth=303
[transpile] WARNING: 0 rotation gates in passthrough output. If you expected Rz gates, check basis_gates and optimization_level.
[2/625] Circuit '1000_101': T=101, qubits=1000


[transpile] Passthrough mode produced 0 rotation gates. The input circuit may already be in a Clifford+T basis, or all rotations were cancelled during optimization. This is expected only if the Hamiltonian has no non-Clifford Trotter terms.


...

[transpile] Passthrough mode: preserving rotation gates (synthesis disabled).
[transpile] Passthrough result (rotations preserved): 0 rotation gate(s) (none)
[transpile] Passthrough done: rotation_gates=0  T_gates=94  depth=282
[transpile] WARNING: 0 rotation gates in passthrough output. If you expected Rz gates, check basis_gates and optimization_level.
[375/625] Circuit '584_94': T=94, qubits=584


[transpile] Passthrough mode produced 0 rotation gates. The input circuit may already be in a Clifford+T basis, or all rotations were cancelled during optimization. This is expected only if the Hamiltonian has no non-Clifford Trotter terms.


[transpile] Passthrough mode: preserving rotation gates (synthesis disabled).
[transpile] Passthrough result (rotations preserved): 0 rotation gate(s) (none)
[transpile] Passthrough done: rotation_gates=0  T_gates=1  depth=3
[transpile] WARNING: 0 rotation gates in passthrough output. If you expected Rz gates, check basis_gates and optimization_level.
[376/625] Circuit '626_1': T=1, qubits=626


[transpile] Passthrough mode produced 0 rotation gates. The input circuit may already be in a Clifford+T basis, or all rotations were cancelled during optimization. This is expected only if the Hamiltonian has no non-Clifford Trotter terms.


[transpile] Passthrough mode: preserving rotation gates (synthesis disabled).
[transpile] Passthrough result (rotations preserved): 0 rotation gate(s) (none)
[transpile] Passthrough done: rotation_gates=0  T_gates=101  depth=303
[transpile] WARNING: 0 rotation gates in passthrough output. If you expected Rz gates, check basis_gates and optimization_level.
[377/625] Circuit '626_101': T=101, qubits=626


### Results table

Each row is one (circuit, estimator) pair.  Empty cells indicate unavailable metrics.


In [ ]:
benchmark_df

,circuit_name,estimator_family,t_count,clifford_count,rotation_count,toffoli_count,measurement_count,runtime_seconds,total_qubits,compute_qubits,factory_qubits,space_time_volume,code_distance,logical_error_rate,error_budget,physical_error_rate,logical_qubits,logical_cycles,factory_count,num_factories
0,1000_1,Azure,1,2,0,0,0,3.6e-06,336711,336651,60,1.212,9,0.007727,0.01,0.001,1000,1,1×T,1
1,1000_1,Qualtran,1,2,0,0,0,2.2e-05,341208,338742,2466,7.507,9,0.03455,0.01,0.001,1000,6.111,"15to1×1 (2,466 qubits)",1
2,1000_101,Azure,101,202,0,0,0,0.0004444,534971,503931,31040,237.7,11,0.008491,0.01,0.001,1000,101,8×T,8
3,1000_101,Qualtran,101,202,0,0,0,0.0004444,535414,506022,29392,237.9,11,0.07034,0.01,0.001,1000,101,"15to1×8 (3,674 each)",8
4,1000_108,Azure,108,216,0,0,0,0.0004752,534971,503931,31040,254.2,11,0.009079,0.01,0.001,1000,108,8×T,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1245,958_80,Qualtran,80,160,0,0,0,0.000352,514602,485210,29392,181.1,11,0.05345,0.01,0.001,958,80,"15to1×8 (3,674 each)",8
1246,958_87,Azure,87,174,0,0,0,0.0003828,514245,483205,31040,196.9,11,0.007089,0.01,0.001,958,87,8×T,8
1247,958_87,Qualtran,87,174,0,0,0,0.0003828,514602,485210,29392,197,11,0.05812,0.01,0.001,958,87,"15to1×8 (3,674 each)",8
1248,958_94,Azure,94,188,0,0,0,0.0004136,514245,483205,31040,212.7,11,0.00766,0.01,0.001,958,94,8×T,8


In [ ]:
from pathlib import Path

results_dir = (
    Path("results")
    / f"{c_count}_{q_count}_{t_count}"
)

results_dir.mkdir(parents=True, exist_ok=True)

benchmark_df.to_csv(
    results_dir / "benchmark_results.csv",
    index=False,
)

In [ ]:
azure = benchmark_df[
    benchmark_df["estimator_family"]=="Azure"
].copy()

qualtran = benchmark_df[
    benchmark_df["estimator_family"]=="Qualtran"
].copy()


ratio_df = pd.merge(
    azure,
    qualtran,
    on=[
        "circuit_name",
        "t_count",
        "logical_qubits",
    ],
    suffixes=("_azure", "_qualtran")
)

print(ratio_df.shape)

(625, 37)


In [ ]:
ratio_df["total_ratio"] = (
    ratio_df["total_qubits_qualtran"]
    /
    ratio_df["total_qubits_azure"]
)


ratio_df["compute_ratio"] = (
    ratio_df["compute_qubits_qualtran"]
    /
    ratio_df["compute_qubits_azure"]
)


ratio_df["factory_ratio"] = (
    ratio_df["factory_qubits_qualtran"]
    /
    ratio_df["factory_qubits_azure"]
)

In [ ]:
ratio_df.to_csv(
    results_dir / "ratio_results.csv",
    index=False,
)

In [ ]:
import json
from dataclasses import asdict

with open(results_dir / "config.json", "w") as f:
    json.dump(asdict(cfg), f, indent=4)